In [1]:
import pandas as pd
from deep.constants import METADATA_DIR, METADATA_FILE, DATA_DIR, SEEDS, MODEL_IMAGE_SIZE, BATCH_SIZE
from deep.modelling.pipiline_utils import split_data

In [2]:
# Load the metadata
data = pd.read_csv(METADATA_FILE)

# Drop unnecessary columns for this problem
data.drop(columns=['phylum','is_animal'], inplace=True)

In [3]:
# Performing the splits
train_df, val_df, test_df = split_data(data, 'family', seed=SEEDS[0])

In [4]:
data

,rare_species_id,family,file_path
0,75fd91cb-2881-41cd-88e6-de451e8b60e2,unionidae,12853737_449393.jpg
1,28c508bc-63ff-4e60-9c8f-1934367e1528,geoemydidae,20969394_793083.jpg
2,00372441-588c-4af8-9665-29bee20822c0,cryptobranchidae,28895411_319982.jpg
3,29cc6040-6af2-49ee-86ec-ab7d89793828,turdidae,29658536_45510188.jpg
4,94004bff-3a33-4758-8125-bf72e6e57eab,indriidae,21252576_7250886.jpg
...,...,...,...
11979,628bf2b4-6ecc-4017-a8e6-4306849e0cfc,emydidae,29972861_1056842.jpg
11980,0ecfdec9-b1cd-4d43-96fc-2f8889ec1ad9,dasyatidae,30134195_52572074.jpg
11981,27fdb1e9-c5fb-459a-8b6a-6fb222b1c512,mustelidae,9474963_46559139.jpg
11982,54894a59-151f-4814-ac32-3a336841e58e,lemuridae,9465817_326525.jpg


In [5]:
upsampled = pd.read_csv(f'{METADATA_DIR}/family_upsample.csv')
upsampled

,rare_species_id,file_path
0,cff98914-6241-46a6-865b-bd77f52ebe34,21938339_912805_accipitridae_rotate_30.jpg
1,0649c1ee-563b-439f-8218-6a107952d9b3,12305608_46561170_acipenseridae_rotate_30.jpg
2,7ce26e92-ca12-423f-8c17-53d538529b10,30098494_205909_acipenseridae_rotate_30.jpg
3,b02da99c-ca3e-4ef0-b480-259969fa0a8f,30098500_205909_acipenseridae_rotate_30.jpg
4,387a82ec-e6da-406c-82ad-28e4b5b10599,29537907_205910_acipenseridae_rotate_30.jpg
...,...,...
13802,789313f7-241e-4bfe-944b-686df6f84b9f,20663357_205714_siluridae_flip_lr.jpg
13803,c7bee7dc-00fc-44c3-8f6e-6612061215c5,28985462_205714_siluridae_bright_plus.jpg
13804,595aa95e-78c3-49df-b421-e8bfd8181a06,28985463_205714_siluridae_bright_plus.jpg
13805,77bd6915-2d97-4412-93c9-8e0f0cbab867,20663365_205714_siluridae_bright_plus.jpg


In [6]:
upsampled = pd.read_csv(f'{METADATA_DIR}/family_upsample.csv')
aux_df2 = upsampled[upsampled['rare_species_id'].isin(train_df['rare_species_id'])]

#Add the new metadata to train_df
train_df = pd.concat([train_df,aux_df2], ignore_index=True, axis=0)

In [7]:
def get_base_image(file_path):
    
    parts = file_path.split('_')
    base_name = '_'.join(parts[:2])  
    base_name = base_name.split('.')[0]

    return base_name

# Build image-to-family map using rows with non-null family values
original_images = train_df[train_df['family'].notna()].copy()
original_images['base_image'] = original_images['file_path'].apply(get_base_image)
image_to_family = dict(zip(original_images['base_image'], original_images['family']))

# Apply base_image extraction to entire dataframe
train_df['base_image'] = train_df['file_path'].apply(get_base_image)

# Fill missing family values
train_df['family'] = train_df.apply(
    lambda row: image_to_family.get(row['base_image'], row['family']) 
    if pd.isna(row['family']) else row['family'], axis=1
)

# Clean up 
train_df.drop(columns=['base_image'], inplace=True)

# Check how many NaNs are still left 
missing_count = train_df['family'].isna().sum()
print(f"Remaining NaN 'family' values: {missing_count}")


Remaining NaN 'family' values: 0


In [8]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.metrics import classification_report
import pandas as pd
import os

In [9]:
base_dir = r"C:\Users\Admin\Desktop\DeepLearning\DeepLearning\data\input_image_directory"

# Update the file paths in both train_df and val_df
train_df['file_path'] = train_df['file_path'].apply(lambda x: os.path.join(base_dir, x))
val_df['file_path'] = val_df['file_path'].apply(lambda x: os.path.join(base_dir, x))

In [10]:
train_df['family'] = train_df['family'].astype(str)
val_df['family'] = val_df['family'].astype(str)
test_df['family'] = test_df['family'].astype(str)

In [11]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator, smart_resize  # type: ignore
base_dir = r"C:\Users\Admin\Desktop\DeepLearning\DeepLearning\data\input_image_directory"
train_datagen = ImageDataGenerator(
        rotation_range=90,
        shear_range=0.2,
        brightness_range=[0.8, 1.2],
        horizontal_flip=True,
        channel_shift_range=30.0,
        zoom_range=(0.8, 1.2),
        fill_mode='nearest',
        preprocessing_function=lambda image: smart_resize(image, size=MODEL_IMAGE_SIZE['efficientnetb4'])
    )
test_datagen = ImageDataGenerator(
        preprocessing_function=lambda image: smart_resize(image, size=MODEL_IMAGE_SIZE['efficientnetb4'])
    )
# Train generator
train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_df,
    directory=base_dir,
    x_col='file_path',
    y_col='family',
    target_size=MODEL_IMAGE_SIZE['efficientnetb4'],
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=SEEDS[0],
    shuffle=True
)

# Validation generator
val_generator = test_datagen.flow_from_dataframe(
    dataframe=val_df,
    directory=base_dir,
    x_col='file_path',
    y_col='family',
    target_size=MODEL_IMAGE_SIZE['efficientnetb4'],
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# Test generator
test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory=base_dir,
    x_col='file_path',
    y_col='family',
    target_size=MODEL_IMAGE_SIZE['efficientnetb4'],
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


Found 17291 validated image filenames belonging to 202 classes.
Found 1439 validated image filenames belonging to 202 classes.


c:\Users\Admin\anaconda\envs\test\Lib\site-packages\keras\src\legacy\preprocessing\image.py:920: UserWarning: Found 232 invalid image filename(s) in x_col="file_path". These filename(s) will be ignored.
  warnings.warn(


Found 2397 validated image filenames belonging to 202 classes.


In [32]:
def get_missing_files(df, base_dir, name=''):
    full_paths = df['file_path'].apply(lambda x: os.path.join(base_dir, x))
    missing_mask = ~full_paths.apply(os.path.exists)
    missing_files = df[missing_mask].copy()
    print(f"\n{name} — Missing Files: {len(missing_files)}")
    if not missing_files.empty:
        print(missing_files[['file_path']].head(10))  # show first 10
    return missing_files

# Check each set
missing_train = get_missing_files(train_df, base_dir, 'Train')
missing_val = get_missing_files(val_df, base_dir, 'Validation')
missing_test = get_missing_files(test_df, base_dir, 'Test')



Train — Missing Files: 232
                                          file_path
8148     21938339_912805_accipitridae_rotate_30.jpg
8150    30098500_205909_acipenseridae_rotate_30.jpg
8151    29537907_205910_acipenseridae_rotate_30.jpg
8152    30098513_205909_acipenseridae_rotate_30.jpg
8153    30098483_205909_acipenseridae_rotate_30.jpg
8154     2746433_205910_acipenseridae_rotate_30.jpg
8157     2746432_205910_acipenseridae_rotate_30.jpg
8159  14185253_46561170_acipenseridae_rotate_30.jpg
8162     2746428_205910_acipenseridae_rotate_30.jpg
8163      9308_46561170_acipenseridae_rotate_30.jpg

Validation — Missing Files: 0

Test — Missing Files: 0


In [ ]:
missing_train

,rare_species_id,family,file_path
8148,cff98914-6241-46a6-865b-bd77f52ebe34,accipitridae,21938339_912805_accipitridae_rotate_30.jpg
8150,b02da99c-ca3e-4ef0-b480-259969fa0a8f,acipenseridae,30098500_205909_acipenseridae_rotate_30.jpg
8151,387a82ec-e6da-406c-82ad-28e4b5b10599,acipenseridae,29537907_205910_acipenseridae_rotate_30.jpg
8152,25543f62-d12b-48df-8a2c-c9bfaf9cd404,acipenseridae,30098513_205909_acipenseridae_rotate_30.jpg
8153,28781188-4cca-4cc5-a9ef-3fa6c5d4dc25,acipenseridae,30098483_205909_acipenseridae_rotate_30.jpg
...,...,...,...
10326,79b65b6c-d1bf-4989-8e0b-173131149c5d,chamaeleonidae,29746944_47044474_chamaeleonidae_sat_plus.jpg
10327,a6db4411-7a2b-4bbd-8e30-91177919c64f,chamaeleonidae,29746928_47044474_chamaeleonidae_sat_plus.jpg
10328,0e51892d-0f56-49a8-9d32-8ed2fe4a7bd5,chamaeleonidae,20589353_1286909_chamaeleonidae_sat_plus.jpg
10329,0176dec1-905c-4f39-9cf4-5fd9a2b7a9b5,chamaeleonidae,20226655_47044474_chamaeleonidae_sat_plus.jpg


In [ ]:
from tensorflow.keras.losses import CategoricalCrossentropy
from deep.preprocess.album_augmenter import compute_effective_class_weights
import tensorflow as tf
from tensorflow.keras.losses import Loss

class CategoricalFocalLoss(Loss):
    def __init__(self, gamma=2.0, alpha=None, from_logits=False, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha
        self.from_logits = from_logits

    def call(self, y_true, y_pred):
        if self.from_logits:
            y_pred = tf.nn.softmax(y_pred)
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)

        cross_entropy = -y_true * tf.math.log(y_pred)
        focal_term = tf.pow(1 - y_pred, self.gamma)

        if self.alpha is not None:
            alpha = tf.constant(self.alpha, dtype=tf.float32)
            alpha_factor = y_true * alpha
            focal_loss = alpha_factor * focal_term * cross_entropy
        else:
            focal_loss = focal_term * cross_entropy

        return tf.reduce_sum(focal_loss, axis=1)

#  Map class weights from labels → indices using the generator
label_map = train_generator.class_indices  # e.g., {'cat': 0, 'dog': 1, ...}
weights_str = compute_effective_class_weights(train_df, 'family')
weights_idx = {label_map[k]: v for k, v in weights_str.items()}

#  Convert to list for alpha
alpha = [weights_idx[i] for i in range(len(weights_idx))]

#  Instantiate focal loss
loss = CategoricalFocalLoss(gamma=5, alpha=alpha)


In [ ]:
def get_fitted_model_metrics(model):
    history = model.history

    # Get the best epoch based on validation loss (or use any other metric)
    best_epoch = np.argmin(history['val_loss'])

    # Loss
    best_train_loss = history['loss'][best_epoch]
    best_val_loss = history['val_loss'][best_epoch]
    print(f"Train Loss: {best_train_loss}\nValidation Loss: {best_val_loss}")

    # Precision (multiclass)
    # Note: Assuming that 'precision' and 'val_precision' are available in history
    # You can access multiclass precision (average across classes)
    best_train_precision = history['precision'][best_epoch]
    best_val_precision = history['val_precision'][best_epoch]
    print(f"Train Precision: {best_train_precision}\nValidation Precision: {best_val_precision}")

    # Alternatively, to get precision per class
    #if 'precision' in history and isinstance(history['precision'], list):
        #for i, class_precision in enumerate(history['precision'][best_epoch]):
            #print(f"Class {i} Precision: {class_precision}")

    return best_val_precision

In [14]:
from tensorflow.keras.layers import Input, Lambda, Dense, GlobalAveragePooling2D, GlobalMaxPooling2D, Concatenate, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import RMSprop

from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, GlobalMaxPooling2D, Concatenate, BatchNormalization, Dropout

def efficient_net_multiclass(
    num_classes: int,  # Number of output classes
    regularizer: bool = False,
    dropout: bool = False
):
    """
    Build and return an EfficientNetB4-based model for multiclass classification.
    Args:
    - num_classes (int): Number of output classes (multiclass)
    - regularizer (bool): Whether to use L2 regularization in the dense layer
    - dropout (bool): Whether to use dropout layer after the dense layer
    
    Returns:
    - multiclass_model (Model): Keras model
    - config (dict): Model configuration
    """
    # Save model configuration
    config = {
        'regularizer': regularizer,
        'dropout': dropout
    }

    # Set the input
    input_tensor = Input(shape=(*MODEL_IMAGE_SIZE["efficientnetb4"], 3))

    # Load the pre-trained model (EfficientNetB4)
    base_model = EfficientNetB4(include_top=False, weights='imagenet', input_tensor=input_tensor)

    # Freeze layers of the pre-trained base model
    base_model.trainable = False

    # Add top layers (Global Average + Global Max Pooling)
    gap = GlobalAveragePooling2D()(base_model.output)
    gmp = GlobalMaxPooling2D()(base_model.output)
    x = Concatenate()([gap, gmp])
    
    # Add regularization or simple dense layer
    if regularizer:
        x = Dense(64, kernel_regularizer=regularizers.l2(0.001), activation='relu')(x)
    else:
        x = Dense(64, activation='relu')(x)
    
    # Add BatchNormalization and Dropout (if required)
    x = BatchNormalization()(x)
    if dropout:
        x = Dropout(0.5)(x)
    
    # Final output layer for multiclass classification (softmax activation)
    output = Dense(num_classes, activation='softmax')(x)

    # Create the model
    multiclass_model = Model(inputs=input_tensor, outputs=output)

    return multiclass_model, config


In [15]:
def plot_metrics(model):
    import matplotlib.pyplot as plt

    history = model.history

    # Defining the variables (multiclass)
    precision = history['precision']
    val_precision = history['val_precision']
    loss = history['loss']
    val_loss = history['val_loss']
    accuracy = history['accuracy']
    val_accuracy = history['val_accuracy']
    epochs = range(1, len(precision) + 1)

    # Plotting Precision (for multiclass, this will plot average precision)
    plt.plot(epochs, precision, 'bo', label='Training Precision')
    plt.plot(epochs, val_precision, 'b', label='Validation Precision')
    plt.title("Training and Validation Precision")
    plt.legend()
    plt.figure()

    # Plotting Accuracy (for multiclass, this is accuracy across all classes)
    plt.plot(epochs, accuracy, 'bo', label='Training Accuracy')
    plt.plot(epochs, val_accuracy, 'b', label='Validation Accuracy')
    plt.title("Training and Validation Accuracy")
    plt.legend()
    plt.figure()

    # Plotting Loss
    plt.plot(epochs, loss, 'bo', label='Training Loss')
    plt.plot(epochs, val_loss, 'b', label='Validation Loss')
    plt.title("Training and Validation Loss")
    plt.legend()

    # Show the plots
    plt.show()


In [16]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import AUC
import numpy as np
num_classes = train_df['family'].nunique()
# Get the model and its configuration
model, config = efficient_net_multiclass(
    num_classes=num_classes,
    regularizer=False,
    dropout=False
)

# Now compile the model
model.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss=loss,  # CategoricalFocalLoss instance or 'categorical_crossentropy'
    metrics=['accuracy', AUC(multi_label=False), 'precision', 'recall']
)

# Fit the model
fitted_model = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    steps_per_epoch=int(np.ceil(len(train_df) / BATCH_SIZE)),
    validation_steps=int(np.ceil(len(val_df) / BATCH_SIZE)),
    class_weight=weights_idx,  # mapped int class weights
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        ReduceLROnPlateau(patience=2, factor=0.5, verbose=1)
    ],
    verbose=1
)

# Get best epoch precision (you might need to adapt this function)
val_precision = get_fitted_model_metrics(fitted_model)

# Plot training metrics
plot_metrics(fitted_model)


c:\Users\Admin\anaconda\envs\test\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
271/274 ━━━━━━━━━━━━━━━━━━━━ 20s 7s/step - accuracy: 0.2084 - auc: 0.7885 - loss: 3.9389 - precision: 0.7380 - recall: 0.0021

c:\Users\Admin\anaconda\envs\test\Lib\site-packages\keras\src\trainers\epoch_iterator.py:107: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


274/274 ━━━━━━━━━━━━━━━━━━━━ 1996s 7s/step - accuracy: 0.2095 - auc: 0.7894 - loss: 3.9317 - precision: 0.7409 - recall: 0.0022 - val_accuracy: 0.3954 - val_auc: 0.9415 - val_loss: 1.8388 - val_precision: 0.9115 - val_recall: 0.0716 - learning_rate: 0.0010
Epoch 2/20
274/274 ━━━━━━━━━━━━━━━━━━━━ 1863s 7s/step - accuracy: 0.5135 - auc: 0.9600 - loss: 1.9305 - precision: 0.9885 - recall: 0.0452 - val_accuracy: 0.5073 - val_auc: 0.9628 - val_loss: 1.2174 - val_precision: 0.9115 - val_recall: 0.1647 - learning_rate: 0.0010
Epoch 3/20
274/274 ━━━━━━━━━━━━━━━━━━━━ 1871s 7s/step - accuracy: 0.6212 - auc: 0.9773 - loss: 1.1414 - precision: 0.9874 - recall: 0.1514 - val_accuracy: 0.5219 - val_auc: 0.9679 - val_loss: 1.1354 - val_precision: 0.8942 - val_recall: 0.2349 - learning_rate: 0.0010
Epoch 4/20
274/274 ━━━━━━━━━━━━━━━━━━━━ 1864s 7s/step - accuracy: 0.6623 - auc: 0.9843 - loss: 0.8854 - precision: 0.9720 - recall: 0.2297 - val_accuracy: 0.5344 - val_auc: 0.9625 - val_loss: 1.1149 - val_pr

TypeError: 'float' object is not iterable

In [17]:
# Save the model architecture
model_json = model.to_json()
with open("model_architecture.json", "w") as json_file:
    json_file.write(model_json)

In [19]:
# Save weights separately
model.save_weights("model_weights.weights.h5")


In [20]:
y_pred_proba = model.predict(test_generator, steps=len(test_generator), verbose=1)

c:\Users\Admin\anaconda\envs\test\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


38/38 ━━━━━━━━━━━━━━━━━━━━ 241s 6s/step


In [21]:
y_pred_classes = np.argmax(y_pred_proba, axis=1)

In [22]:
y_true_classes = test_generator.classes  # integers
class_indices = test_generator.class_indices
class_labels = list(class_indices.keys())

In [25]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# Accuracy
print("Accuracy:", accuracy_score(y_true_classes, y_pred_classes))

# F1 Scores
f1_macro = f1_score(y_true_classes, y_pred_classes, average='macro')
f1_weighted = f1_score(y_true_classes, y_pred_classes, average='weighted')
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")

# Precision
precision_macro = precision_score(y_true_classes, y_pred_classes, average='macro')
precision_weighted = precision_score(y_true_classes, y_pred_classes, average='weighted')
print(f"Precision (Macro): {precision_macro:.4f}")
print(f"Precision (Weighted): {precision_weighted:.4f}")

# Recall
recall_macro = recall_score(y_true_classes, y_pred_classes, average='macro')
recall_weighted = recall_score(y_true_classes, y_pred_classes, average='weighted')
print(f"Recall (Macro): {recall_macro:.4f}")
print(f"Recall (Weighted): {recall_weighted:.4f}")

# Classification report
print("\nClassification Report:")
print(classification_report(y_true_classes, y_pred_classes, target_names=class_labels))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_true_classes, y_pred_classes))


Accuracy: 0.672090112640801
F1 Score (Macro): 0.6656
F1 Score (Weighted): 0.6722
Precision (Macro): 0.6860
Precision (Weighted): 0.7029
Recall (Macro): 0.6817
Recall (Weighted): 0.6721

Classification Report:
                   precision    recall  f1-score   support

     accipitridae       0.73      0.67      0.70        24
    acipenseridae       0.47      0.78      0.58        18
      acroporidae       0.71      0.71      0.71        42
         agamidae       0.54      0.58      0.56        12
      agariciidae       0.62      0.33      0.43        24
        albulidae       0.33      0.33      0.33         6
      alcedinidae       0.83      0.83      0.83         6
    alligatoridae       0.62      0.83      0.71         6
        alopiidae       0.64      0.58      0.61        12
   ambystomatidae       0.47      0.67      0.55        12
         anatidae       0.85      0.64      0.73        36
         anguidae       0.70      0.58      0.64        12
          aotidae      

In [28]:
train_df['file_path'] = train_df['file_path'].str.replace(base_dir + os.sep, '', regex=False)
val_df['file_path'] = val_df['file_path'].str.replace(base_dir + os.sep, '', regex=False)


In [29]:
train_df

,rare_species_id,family,file_path
0,002cf717-3a67-42c8-b855-cf8b2c4d6bd7,syngnathidae,2718903_46567771.jpg
1,9bed8543-3578-4312-8dc5-1c74f57006d3,iguanidae,28478138_793232.jpg
2,d15bc487-08c4-4a64-b8ee-b7d439f5afe7,pardalotidae,28243834_45518587.jpg
3,a5aec2cf-749d-4b4a-9b7c-03cbca6425e6,hylobatidae,28286079_4454250.jpg
4,54aba4e2-4ea5-4fd7-ae22-0f92db1abdab,faviidae,28170659_45276848.jpg
...,...,...,...
17518,16bb0ba2-fdbc-4f44-aa1a-e61fab60a84a,siluridae,24576328_205714_siluridae_flip_lr.jpg
17519,595aa95e-78c3-49df-b421-e8bfd8181a06,siluridae,28985463_205714_siluridae_flip_lr.jpg
17520,7a8f294a-bbb3-4ead-9e7e-dfa30f3c3ba3,siluridae,20663363_205714_siluridae_flip_lr.jpg
17521,c7bee7dc-00fc-44c3-8f6e-6612061215c5,siluridae,28985462_205714_siluridae_bright_plus.jpg


In [27]:
test_df

,rare_species_id,family,file_path
1944,7f077bfd-6389-4101-92c8-97ee5bfdff0d,cricetidae,29389823_1179513.jpg
7520,3caf5d36-8244-42d9-bc0b-f091407adcc3,lobophylliidae,28422156_46545536.jpg
8775,b763f4ec-c379-4876-b65b-6f6be901656c,plethodontidae,14053934_336170.jpg
3783,ea50f1b6-8431-4ec9-9143-3677f0c7fa27,bufonidae,14154469_130707.jpg
6907,09d17581-1be6-4a82-9d08-541433d1a10c,cryptobranchidae,28895399_319982.jpg
...,...,...,...
10202,9f9a29d5-e948-45ed-8ba7-f09f5d8b0bba,gymnuridae,22078989_46560977.jpg
1263,0ec90545-271c-4630-b247-7e44724e7d96,dactyloidae,28674263_795876.jpg
10723,19958c19-8608-4318-8519-38c927c7175a,carettochelyidae,22105011_1056984.jpg
10150,b0856ed4-b273-4fd5-a19e-23ab43b90e86,serranidae,20080362_46579618.jpg


In [31]:
# Convert to sets
train_paths = set(train_df['file_path'])
val_paths = set(val_df['file_path'])
test_paths = set(test_df['file_path'])

# Check intersections
common_train_val = train_paths & val_paths
common_train_test = train_paths & test_paths
common_val_test = val_paths & test_paths
common_all = train_paths & val_paths & test_paths

# Report
print(f"Common between train and val: {len(common_train_val)}")
print(f"Common between train and test: {len(common_train_test)}")
print(f"Common between val and test: {len(common_val_test)}")
print(f"Common in all three: {len(common_all)}")


Common between train and val: 0
Common between train and test: 0
Common between val and test: 0
Common in all three: 0
